In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [1]:
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.3 MB/s eta 0:00:00


# **Code Ready**

In [2]:
!git clone https://github.com/facebookresearch/dino

Cloning into 'dino'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 175 (delta 57), reused 55 (delta 54), pack-reused 106
Receiving objects: 100% (175/175), 24.44 MiB | 6.21 MiB/s, done.
Resolving deltas: 100% (109/109), done.


In [3]:
%cd dino

/content/dino


# **Data Ready**

In [4]:
!gdown 1BRccmUBnxhbthHL0bxVz1nLD7OSbZ3r2
!unzip train.zip
!rm train.zip
!rm -r ./train/1
!rm -r ./train/2
!rm -r ./train/4
!rm -r ./train/6

Streaming output truncated to the last 5000 lines.
  inflating: train/6/s6_39679_s6_40_105.png  
  inflating: train/6/s6_39684_s6_40_105.png  
  inflating: train/6/s6_39688_s6_40_105.png  
  inflating: train/6/s6_39690_s6_40_105.png  
  inflating: train/6/s6_39694_s6_40_105.png  
  inflating: train/6/s6_39699_s6_40_105.png  
  inflating: train/6/s6_39709_s6_40_105.png  
  inflating: train/6/s6_39711_s6_40_105.png  
  inflating: train/6/s6_39718_s6_40_105.png  
  inflating: train/6/s6_39729_s6_40_105.png  
  inflating: train/6/s6_39733_s6_40_105.png  
  inflating: train/6/s6_39748_s6_40_144.png  
  inflating: train/6/s6_39749_s6_40_144.png  
  inflating: train/6/s6_39761_s6_40_144.png  
  inflating: train/6/s6_39768_s6_40_144.png  
  inflating: train/6/s6_39770_s6_40_144.png  
  inflating: train/6/s6_39776_s6_40_144.png  
  inflating: train/6/s6_39782_s6_40_144.png  
  inflating: train/6/s6_39783_s6_40_144.png  
  inflating: train/6/s6_39787_s6_40_144.png  
  inflating: train/6/s6_39790

In [5]:
!gdown 1Y-hVa9N-AyabzIc-KKqr5TTwH-tdrwin
!unzip bracs_ood_test.zip
!mv final_test test
!rm bracs_ood_test.zip
!rm -r ./test/s1
!rm -r ./test/s2
!rm -r ./test/s4
!rm -r ./test/s6
!mv ./test/s0 ./test/0
!mv ./test/s3 ./test/3
!mv ./test/s5 ./test/5

Streaming output truncated to the last 5000 lines.
  inflating: final_test/s2/s2_1016_s2_1_100.png  
  inflating: final_test/s2/s2_1020_s2_1_100.png  
  inflating: final_test/s2/s2_1022_s2_1_100.png  
  inflating: final_test/s2/s2_1023_s2_1_100.png  
  inflating: final_test/s2/s2_1028_s2_1_100.png  
  inflating: final_test/s2/s2_1034_s2_1_112.png  
  inflating: final_test/s2/s2_1040_s2_1_13.png  
  inflating: final_test/s2/s2_1048_s2_1_13.png  
  inflating: final_test/s2/s2_1051_s2_1_13.png  
  inflating: final_test/s2/s2_1053_s2_1_13.png  
  inflating: final_test/s2/s2_1054_s2_1_13.png  
  inflating: final_test/s2/s2_1055_s2_1_13.png  
  inflating: final_test/s2/s2_1061_s2_1_13.png  
  inflating: final_test/s2/s2_1062_s2_1_13.png  
  inflating: final_test/s2/s2_1064_s2_1_13.png  
  inflating: final_test/s2/s2_1067_s2_1_13.png  
  inflating: final_test/s2/s2_1071_s2_1_13.png  
  inflating: final_test/s2/s2_107_s2_0_13.png  
  inflating: final_test/s2/s2_1080_s2_1_13.png  
  inflating: 

# **Validation and get embedding**

<font color="yellow">
Here put your final trained checkpoint in model directory
</font>

In [6]:
%mkdir model

In [7]:
!gdown -O ./model/checkpoint.pth 1-Gc1BjleLfN2Ue1Gym3jxag1xewRKYlL

Downloading...
From: https://drive.google.com/uc?id=1-Gc1BjleLfN2Ue1Gym3jxag1xewRKYlL
To: /content/dino/model/checkpoint.pth
100% 704M/704M [00:11<00:00, 63.4MB/s]


In [8]:
%mkdir data2
%cp -r ./train/ ./data2/train
%cp -r ./test/ ./data2/val

<font color="red">
don't run below cell
</font>
<br>
<font color="yellow">
change eval_knn.py file with below code to save embedding, label and file name
</font>

In [ ]:
# Copyright (c) Facebook, Inc. and its affiliates.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
import os
import sys
import argparse
import json

import torch
from torch import nn
import torch.distributed as dist
import torch.backends.cudnn as cudnn
from torchvision import datasets
from torchvision import transforms as pth_transforms
from torchvision import models as torchvision_models

import utils
import vision_transformer as vits


def extract_feature_pipeline(args):
    # ============ preparing data ... ============
    transform = pth_transforms.Compose([
        pth_transforms.Resize(256, interpolation=3),
        pth_transforms.CenterCrop(224),
        pth_transforms.ToTensor(),
        pth_transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])
    dataset_train = ReturnIndexDataset(os.path.join(args.data_path, "train"), transform=transform)
    dataset_val = ReturnIndexDataset(os.path.join(args.data_path, "val"), transform=transform)
    sampler = torch.utils.data.DistributedSampler(dataset_train, shuffle=False)
    data_loader_train = torch.utils.data.DataLoader(
        dataset_train,
        sampler=sampler,
        batch_size=args.batch_size_per_gpu,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=False,
    )
    data_loader_val = torch.utils.data.DataLoader(
        dataset_val,
        batch_size=args.batch_size_per_gpu,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=False,
    )
    print(f"Data loaded with {len(dataset_train)} train and {len(dataset_val)} val imgs.")

    # ============ building network ... ============
    if "vit" in args.arch:
        model = vits.__dict__[args.arch](patch_size=args.patch_size, num_classes=0)
        print(f"Model {args.arch} {args.patch_size}x{args.patch_size} built.")
    elif "xcit" in args.arch:
        model = torch.hub.load('facebookresearch/xcit:main', args.arch, num_classes=0)
    elif args.arch in torchvision_models.__dict__.keys():
        model = torchvision_models.__dict__[args.arch](num_classes=0)
        model.fc = nn.Identity()
    else:
        print(f"Architecture {args.arch} non supported")
        sys.exit(1)
    model.cuda()
    utils.load_pretrained_weights(model, args.pretrained_weights, args.checkpoint_key, args.arch, args.patch_size)
    model.eval()

    # ============ extract features ... ============
    print("Extracting features for train set...")
    train_features, train_file_names = extract_features(model, data_loader_train, args.use_cuda)
    print("Extracting features for val set...")
    test_features, test_file_names = extract_features(model, data_loader_val, args.use_cuda)

    if utils.get_rank() == 0:
        train_features = nn.functional.normalize(train_features, dim=1, p=2)
        test_features = nn.functional.normalize(test_features, dim=1, p=2)

    train_labels = torch.tensor([s[-1] for s in dataset_train.samples]).long()
    test_labels = torch.tensor([s[-1] for s in dataset_val.samples]).long()
    # save features and labels
    all_file_names = {"train_file_names": train_file_names, "test_file_names": test_file_names}
    if args.dump_features and dist.get_rank() == 0:
        torch.save(train_features.cpu(), os.path.join(args.dump_features, "trainfeat.pth"))
        torch.save(test_features.cpu(), os.path.join(args.dump_features, "testfeat.pth"))
        torch.save(train_labels.cpu(), os.path.join(args.dump_features, "trainlabels.pth"))
        torch.save(test_labels.cpu(), os.path.join(args.dump_features, "testlabels.pth"))
        with open(os.path.join(args.dump_features, "file_names.json"), 'w') as f:
          json.dump(all_file_names, f)
    return train_features, test_features, train_labels, test_labels, train_file_names, test_file_names


@torch.no_grad()
def extract_features(model, data_loader, use_cuda=True, multiscale=False):
    metric_logger = utils.MetricLogger(delimiter="  ")
    features = None
    all_file_names = list()
    for samples, index, file_names in metric_logger.log_every(data_loader, 10):
        samples = samples.cuda(non_blocking=True)
        index = index.cuda(non_blocking=True)
        all_file_names += file_names
        if multiscale:
            feats = utils.multi_scale(samples, model)
        else:
            feats = model(samples).clone()

        # init storage feature matrix
        if dist.get_rank() == 0 and features is None:
            features = torch.zeros(len(data_loader.dataset), feats.shape[-1])
            if use_cuda:
                features = features.cuda(non_blocking=True)
            print(f"Storing features into tensor of shape {features.shape}")

        # get indexes from all processes
        y_all = torch.empty(dist.get_world_size(), index.size(0), dtype=index.dtype, device=index.device)
        y_l = list(y_all.unbind(0))
        y_all_reduce = torch.distributed.all_gather(y_l, index, async_op=True)
        y_all_reduce.wait()
        index_all = torch.cat(y_l)

        # share features between processes
        feats_all = torch.empty(
            dist.get_world_size(),
            feats.size(0),
            feats.size(1),
            dtype=feats.dtype,
            device=feats.device,
        )
        output_l = list(feats_all.unbind(0))
        output_all_reduce = torch.distributed.all_gather(output_l, feats, async_op=True)
        output_all_reduce.wait()

        # update storage feature matrix
        if dist.get_rank() == 0:
            if use_cuda:
                features.index_copy_(0, index_all, torch.cat(output_l))
            else:
                features.index_copy_(0, index_all.cpu(), torch.cat(output_l).cpu())
    return features, all_file_names


@torch.no_grad()
def knn_classifier(train_features, train_labels, test_features, test_labels, k, T, num_classes=1000):
    top1, top5, total = 0.0, 0.0, 0
    train_features = train_features.t()
    num_test_images, num_chunks = test_labels.shape[0], 100
    imgs_per_chunk = num_test_images // num_chunks
    retrieval_one_hot = torch.zeros(k, num_classes).to(train_features.device)
    for idx in range(0, num_test_images, imgs_per_chunk):
        # get the features for test images
        features = test_features[
            idx : min((idx + imgs_per_chunk), num_test_images), :
        ]
        targets = test_labels[idx : min((idx + imgs_per_chunk), num_test_images)]
        batch_size = targets.shape[0]

        # calculate the dot product and compute top-k neighbors
        similarity = torch.mm(features, train_features)
        distances, indices = similarity.topk(k, largest=True, sorted=True)
        candidates = train_labels.view(1, -1).expand(batch_size, -1)
        retrieved_neighbors = torch.gather(candidates, 1, indices)

        retrieval_one_hot.resize_(batch_size * k, num_classes).zero_()
        retrieval_one_hot.scatter_(1, retrieved_neighbors.view(-1, 1), 1)
        distances_transform = distances.clone().div_(T).exp_()
        probs = torch.sum(
            torch.mul(
                retrieval_one_hot.view(batch_size, -1, num_classes),
                distances_transform.view(batch_size, -1, 1),
            ),
            1,
        )
        _, predictions = probs.sort(1, True)

        # find the predictions that match the target
        correct = predictions.eq(targets.data.view(-1, 1))
        top1 = top1 + correct.narrow(1, 0, 1).sum().item()
        top5 = top5 + correct.narrow(1, 0, min(5, k)).sum().item()  # top5 does not make sense if k < 5
        total += targets.size(0)
    top1 = top1 * 100.0 / total
    top5 = top5 * 100.0 / total
    return top1, top5


class ReturnIndexDataset(datasets.ImageFolder):
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        image = self.loader(path)
        filename = os.path.basename(path)

        if self.transform is not None:
          image = self.transform(image)

        if self.target_transform is not None:
          target = self.target_transform(target)

        return image, idx, filename


if __name__ == '__main__':
    parser = argparse.ArgumentParser('Evaluation with weighted k-NN on ImageNet')
    parser.add_argument('--batch_size_per_gpu', default=128, type=int, help='Per-GPU batch-size')
    parser.add_argument('--nb_knn', default=[10, 20, 100, 200], nargs='+', type=int,
        help='Number of NN to use. 20 is usually working the best.')
    parser.add_argument('--temperature', default=0.07, type=float,
        help='Temperature used in the voting coefficient')
    parser.add_argument('--pretrained_weights', default='', type=str, help="Path to pretrained weights to evaluate.")
    parser.add_argument('--use_cuda', default=True, type=utils.bool_flag,
        help="Should we store the features on GPU? We recommend setting this to False if you encounter OOM")
    parser.add_argument('--arch', default='vit_small', type=str, help='Architecture')
    parser.add_argument('--patch_size', default=16, type=int, help='Patch resolution of the model.')
    parser.add_argument("--checkpoint_key", default="teacher", type=str,
        help='Key to use in the checkpoint (example: "teacher")')
    parser.add_argument('--dump_features', default=None,
        help='Path where to save computed features, empty for no saving')
    parser.add_argument('--load_features', default=None, help="""If the features have
        already been computed, where to find them.""")
    parser.add_argument('--num_workers', default=10, type=int, help='Number of data loading workers per GPU.')
    parser.add_argument("--dist_url", default="env://", type=str, help="""url used to set up
        distributed training; see https://pytorch.org/docs/stable/distributed.html""")
    parser.add_argument("--local_rank", default=0, type=int, help="Please ignore and do not set this argument.")
    parser.add_argument('--data_path', default='/path/to/imagenet/', type=str)
    args = parser.parse_args()

    utils.init_distributed_mode(args)
    print("git:\n  {}\n".format(utils.get_sha()))
    print("\n".join("%s: %s" % (k, str(v)) for k, v in sorted(dict(vars(args)).items())))
    cudnn.benchmark = True

    if args.load_features:
        train_features = torch.load(os.path.join(args.load_features, "trainfeat.pth"))
        test_features = torch.load(os.path.join(args.load_features, "testfeat.pth"))
        train_labels = torch.load(os.path.join(args.load_features, "trainlabels.pth"))
        test_labels = torch.load(os.path.join(args.load_features, "testlabels.pth"))
    else:
        # need to extract features !
        train_features, test_features, train_labels, test_labels, train_file_names, test_file_names = extract_feature_pipeline(args)

    if utils.get_rank() == 0:
        if args.use_cuda:
            train_features = train_features.cuda()
            test_features = test_features.cuda()
            train_labels = train_labels.cuda()
            test_labels = test_labels.cuda()

        print("Features are ready!\nStart the k-NN classification.")
        for k in args.nb_knn:
            top1, top5 = knn_classifier(train_features, train_labels,
                test_features, test_labels, k, args.temperature)
            print(f"{k}-NN classifier result: Top1: {top1}, Top5: {top5}")
    dist.barrier()

In [9]:
!python eval_knn.py --pretrained_weights ./model/checkpoint.pth --patch_size 8 --checkpoint_key teacher --dump_features ./model/ --num_workers 1 --data_path ./data2/

Will run the code on one GPU.
| distributed init (rank 0): env://
git:
  sha: 7c446df5b9f45747937fb0d72314eb9f7b66930a, status: has uncommited changes, branch: main

arch: vit_small
batch_size_per_gpu: 128
checkpoint_key: teacher
data_path: ./data2/
dist_url: env://
dump_features: ./model/
gpu: 0
load_features: None
local_rank: 0
nb_knn: [10, 20, 100, 200]
num_workers: 1
patch_size: 8
pretrained_weights: ./model/checkpoint.pth
rank: 0
temperature: 0.07
use_cuda: True
world_size: 1
Data loaded with 29732 train and 3000 val imgs.
Model vit_small 8x8 built.
Take key teacher in provided checkpoint dict
Pretrained weights found at ./model/checkpoint.pth and loaded with msg: _IncompatibleKeys(missing_keys=[], unexpected_keys=['head.mlp.0.weight', 'head.mlp.0.bias', 'head.mlp.2.weight', 'head.mlp.2.bias', 'head.mlp.4.weight', 'head.mlp.4.bias', 'head.last_layer.weight_g', 'head.last_layer.weight_v'])
Extracting features for train set...
Storing features into tensor of shape torch.Size([29732,